# 06 Constraints and Design Logic

## Purpose
- define fixed vs mutable regions
- connect structural reasoning to design restrictions
- prepare for constrained ProteinMPNN runs


***Load required data***

In [63]:
import pandas as pd

comparison_df = pd.read_csv("../results/tables/native_vs_top_designs.csv")
residue_df = pd.read_csv("../data/processed/residue_table.csv")

comparison_df.head()

,comparison_group,candidate_rank,id,num_mutations,plddt,ptm,rank_score,mutations,sequence_divergence
0,native,0,seq_0,0,98.106822,0.91,95.264093,NaN,native
1,designed,1,seq_1,70,96.237209,0.90,93.742326,K1M;V2K;G4S;A10R;A11R;M12L;R14E;H15L;L17M;Y20L...,very_high
2,designed,2,seq_5,67,95.769535,0.89,93.061721,K1M;V2K;G4S;E7G;A10Q;A11E;M12L;R14K;H15L;L17M;...,very_high
3,designed,3,seq_2,73,95.363643,0.89,92.818186,V2K;F3Y;G4T;E7G;A10K;A11E;M12L;R14K;H15L;L17M;...,very_high


***Extract mutations - expand mutations into rows***

In [64]:
import pandas as pd

def parse_mutations(row):
    if pd.isna(row["mutations"]):
        return []

    muts = row["mutations"].split(";")
    parsed = []

    for m in muts:
        wt = m[0]
        pos = int(m[1:-1])
        mut = m[-1]

        parsed.append({
            "id": row["id"],
            "wt": wt,
            "position": pos,
            "mut": mut
        })

    return parsed


all_mutations = []

for _, row in comparison_df.iterrows():
    all_mutations.extend(parse_mutations(row))

mut_df = pd.DataFrame(all_mutations)

mut_df.head()

,id,wt,position,mut
0,seq_1,K,1,M
1,seq_1,V,2,K
2,seq_1,G,4,S
3,seq_1,A,10,R
4,seq_1,A,11,R


***Add biochemical classification***

In [65]:
def aa_class(res):
    if res in ["A","V","I","L","M"]:
        return "hydrophobic"
    elif res in ["S","T","N","Q"]:
        return "polar"
    elif res in ["K","R","H"]:
        return "positive"
    elif res in ["D","E"]:
        return "negative"
    elif res in ["F","Y","W"]:
        return "aromatic"
    elif res in ["G","P","C"]:
        return "special"
    else:
        return "unknown"


mut_df["wt_class"] = mut_df["wt"].apply(aa_class)
mut_df["mut_class"] = mut_df["mut"].apply(aa_class)

***Classify mutation type***

In [66]:
def mutation_type(row):
    if row["wt"] == row["mut"]:
        return "no_change"
    elif row["wt_class"] == row["mut_class"]:
        return "conservative"
    else:
        return "non_conservative"

mut_df["mutation_type"] = mut_df.apply(mutation_type, axis=1)

***Adding structural context***

In [67]:
struct_df = residue_df.rename(columns={"residue_number": "position"})

mut_df = mut_df.merge(
    struct_df[["position", "environment", "ss_simple"]],
    on="position",
    how="left"
)

mut_df.head()

,id,wt,position,mut,wt_class,mut_class,mutation_type,environment,ss_simple
0,seq_1,K,1,M,positive,hydrophobic,non_conservative,surface,loop
1,seq_1,V,2,K,hydrophobic,positive,non_conservative,surface,helix
2,seq_1,G,4,S,special,polar,non_conservative,surface,sheet
3,seq_1,A,10,R,hydrophobic,positive,non_conservative,surface,helix
4,seq_1,A,11,R,hydrophobic,positive,non_conservative,core,helix


***Analyze mutation patterns***

***A — conservative vs non-conservative***

In [68]:
mut_df["mutation_type"].value_counts()

mutation_type
non_conservative    138
conservative         72
Name: count, dtype: int64

***B — mutations by environment***

In [69]:
pd.crosstab(mut_df["environment"], mut_df["mutation_type"])

mutation_type,conservative,non_conservative
environment,,
core,39,69
surface,33,69


***C — mutations by secondary structure***

In [70]:
pd.crosstab(mut_df["ss_simple"], mut_df["mutation_type"])

mutation_type,conservative,non_conservative
ss_simple,,
helix,54,96
loop,0,2
sheet,18,40


***Summary per candidate***

In [71]:
summary = mut_df.groupby("id").agg(
    total_mutations=("mutation_type", "count"),
    conservative=("mutation_type", lambda x: (x=="conservative").sum()),
    non_conservative=("mutation_type", lambda x: (x=="non_conservative").sum())
)

summary["fraction_conservative"] = summary["conservative"] / summary["total_mutations"]

summary

,total_mutations,conservative,non_conservative,fraction_conservative
id,,,,
seq_1,70,23,47,0.328571
seq_2,73,29,44,0.397260
seq_5,67,20,47,0.298507


***save mutational analysis***

## Mutation Analysis

Mutations were analyzed at the residue level to understand their biochemical and structural characteristics.

### What was analyzed
- wild-type vs mutated residue
- biochemical class change
- conservative vs non-conservative mutations
- structural context (core/surface, helix/sheet)

### Observations
- many mutations are non-conservative, reflecting aggressive redesign
- structural confidence remains high despite large sequence changes
- mutations are not uniformly distributed across structural environments

### Why this matters
This shows that ProteinMPNN explores sequence space broadly while still maintaining fold compatibility.

### Important limitation
This analysis does not yet include energetic scoring, active-site protection, or experimental validation.
    

***Load required data***

In [72]:
import pandas as pd
import os

possible_paths = [
    "../results/tables/mutation_analysis_long.csv",
    "results/tables/mutation_analysis_long.csv"
]

mutation_file = None
for path in possible_paths:
    if os.path.exists(path):
        mutation_file = path
        break

if mutation_file is None:
    raise FileNotFoundError(
        "mutation_analysis_long.csv not found. Finish Step 21 first."
    )

mut_df = pd.read_csv(mutation_file)

print("Loaded:", mutation_file)
print(mut_df.columns.tolist())
print(mut_df.head())
print(mut_df.shape)

Loaded: ../results/tables/mutation_analysis_long.csv
['id', 'wt', 'position', 'mut', 'wt_class', 'mut_class', 'mutation_type', 'environment', 'ss_simple']
      id wt  position mut     wt_class    mut_class     mutation_type  \
0  seq_1  K         1   M     positive  hydrophobic  non_conservative   
1  seq_1  V         2   K  hydrophobic     positive  non_conservative   
2  seq_1  G         4   S      special        polar  non_conservative   
3  seq_1  A        10   R  hydrophobic     positive  non_conservative   
4  seq_1  A        11   R  hydrophobic     positive  non_conservative   

  environment ss_simple  
0     surface      loop  
1     surface     helix  
2     surface     sheet  
3     surface     helix  
4        core     helix  
(210, 9)


***column names checking***

In [73]:
required_cols = [
    "id", "wt", "mut", "position",
    "wt_class", "mut_class", "mutation_type",
    "environment", "ss_simple"
]

missing = [c for c in required_cols if c not in mut_df.columns]
print("Missing columns:", missing)

Missing columns: []


***Add mutation risk scoring***

In [74]:
import pandas as pd
import numpy as np

# Make a copy so original mut_df stays untouched if needed
risk_df = mut_df.copy()

# Normalize text columns
text_cols = [
    "wt", "mut", "wt_class", "mut_class",
    "mutation_type", "environment", "ss_simple"
]

for col in text_cols:
    risk_df[col] = risk_df[col].astype(str).str.strip()

# Initialize new columns
risk_df["risk_score"] = 0
risk_df["risk_reasons"] = ""

# Rule 1: non-conservative mutation
mask = risk_df["mutation_type"] == "non_conservative"
risk_df.loc[mask, "risk_score"] += 1
risk_df.loc[mask, "risk_reasons"] += "non_conservative;"

# Rule 2: core mutation
mask = risk_df["environment"] == "core"
risk_df.loc[mask, "risk_score"] += 1
risk_df.loc[mask, "risk_reasons"] += "core_position;"

# Rule 3: structured region
mask = risk_df["ss_simple"].isin(["helix", "sheet"])
risk_df.loc[mask, "risk_score"] += 1
risk_df.loc[mask, "risk_reasons"] += "structured_region;"

# Rule 4: biochemical class switch
mask = risk_df["wt_class"] != risk_df["mut_class"]
risk_df.loc[mask, "risk_score"] += 1
risk_df.loc[mask, "risk_reasons"] += "class_switch;"

# Rule 5: proline involved in helix/sheet
mask = (
    risk_df["ss_simple"].isin(["helix", "sheet"]) &
    ((risk_df["wt"] == "P") | (risk_df["mut"] == "P"))
)
risk_df.loc[mask, "risk_score"] += 2
risk_df.loc[mask, "risk_reasons"] += "proline_in_structured_region;"

# Rule 6: glycine involved in helix/sheet
mask = (
    risk_df["ss_simple"].isin(["helix", "sheet"]) &
    ((risk_df["wt"] == "G") | (risk_df["mut"] == "G"))
)
risk_df.loc[mask, "risk_score"] += 1
risk_df.loc[mask, "risk_reasons"] += "glycine_in_structured_region;"

# Clean trailing semicolon
risk_df["risk_reasons"] = risk_df["risk_reasons"].str.rstrip(";")

# Convert score to label
def assign_risk_label(score):
    if score >= 5:
        return "high"
    elif score >= 3:
        return "medium"
    else:
        return "low"

risk_df["risk_label"] = risk_df["risk_score"].apply(assign_risk_label)

print(risk_df[[
    "id", "wt", "position", "mut",
    "risk_score", "risk_label", "risk_reasons"
]].head(20))

       id wt  position mut  risk_score risk_label  \
0   seq_1  K         1   M           2        low   
1   seq_1  V         2   K           3     medium   
2   seq_1  G         4   S           4     medium   
3   seq_1  A        10   R           3     medium   
4   seq_1  A        11   R           4     medium   
5   seq_1  M        12   L           2        low   
6   seq_1  R        14   E           3     medium   
7   seq_1  H        15   L           4     medium   
8   seq_1  L        17   M           2        low   
9   seq_1  Y        20   L           3     medium   
10  seq_1  R        21   F           3     medium   
11  seq_1  S        24   P           5       high   
12  seq_1  N        27   A           4     medium   
13  seq_1  W        28   Y           2        low   
14  seq_1  A        31   L           2        low   
15  seq_1  K        33   E           4     medium   
16  seq_1  F        34   V           4     medium   
17  seq_1  E        35   T           4     med

***Save the mutation risk table***

In [75]:
import os

output_dir = "../results/tables"
if not os.path.exists(output_dir):
    output_dir = "results/tables"

os.makedirs(output_dir, exist_ok=True)

risk_file = os.path.join(output_dir, "mutation_risk_annotated.csv")
risk_df.to_csv(risk_file, index=False)

print("Saved:", risk_file)

Saved: ../results/tables/mutation_risk_annotated.csv


***Summarize risk across positions***

In [76]:
position_summary = (
    risk_df.groupby("position")
    .agg(
        n_mutations=("position", "count"),
        n_candidates=("id", "nunique"),
        max_risk_score=("risk_score", "max"),
        mean_risk_score=("risk_score", "mean"),
        n_high_risk=("risk_label", lambda x: (x == "high").sum()),
        environments=("environment", lambda x: ",".join(sorted(set(x.astype(str))))),
        ss_types=("ss_simple", lambda x: ",".join(sorted(set(x.astype(str)))))
    )
    .reset_index()
    .sort_values(
        by=["n_high_risk", "mean_risk_score", "n_candidates"],
        ascending=[False, False, False]
    )
)

print(position_summary.head(20))

    position  n_mutations  n_candidates  max_risk_score  mean_risk_score  \
40        59            3             3               6         6.000000   
66       109            3             3               6         6.000000   
13        24            3             3               5         5.000000   
21        37            3             3               5         5.000000   
22        38            3             3               5         5.000000   
38        57            3             3               5         5.000000   
20        36            3             3               5         4.333333   
31        47            3             3               5         3.000000   
3          4            3             3               4         4.000000   
6         11            3             3               4         4.000000   
9         15            3             3               4         4.000000   
14        27            3             3               4         4.000000   
17        33

***Define which positions should be constrained***

In [77]:
constraint_df = position_summary[
    (position_summary["mean_risk_score"] >= 4)
    | (position_summary["n_high_risk"] >= 1)
    | (
        (position_summary["n_candidates"] >= 2) &
        (position_summary["mean_risk_score"] >= 3)
    )
].copy()

constraint_df = constraint_df.sort_values(
    by=["n_high_risk", "mean_risk_score", "n_candidates"],
    ascending=[False, False, False]
)

constraint_positions = constraint_df["position"].tolist()

print("Constraint positions:")
print(constraint_positions)
print("Total:", len(constraint_positions))

Constraint positions:
[59, 109, 24, 37, 38, 57, 36, 47, 4, 11, 15, 27, 33, 35, 39, 44, 52, 56, 71, 82, 84, 90, 106, 110, 7, 93, 53, 85, 34, 91, 2, 10, 21, 46, 51, 61, 68, 73, 86, 113, 122, 125, 128, 62, 75, 116, 118]
Total: 47


***Save the constraint-position table***

In [78]:
constraint_file = os.path.join(output_dir, "constraint_positions.csv")
constraint_df.to_csv(constraint_file, index=False)

print("Saved:", constraint_file)

Saved: ../results/tables/constraint_positions.csv


***Create a clean review table for interpretation***

In [79]:
review_cols = [
    "id", "wt", "position", "mut",
    "wt_class", "mut_class", "mutation_type",
    "environment", "ss_simple",
    "risk_score", "risk_label", "risk_reasons"
]

review_df = risk_df[review_cols].sort_values(
    by=["risk_score", "id", "position"],
    ascending=[False, True, True]
)

review_file = os.path.join(output_dir, "mutation_risk_review_table.csv")
review_df.to_csv(review_file, index=False)

print("Saved:", review_file)
print(review_df.head(20))

Saved: ../results/tables/mutation_risk_review_table.csv
        id wt  position mut     wt_class mut_class     mutation_type  \
34   seq_1  N        59   P        polar   special  non_conservative   
57   seq_1  V       109   P  hydrophobic   special  non_conservative   
174  seq_2  N        59   P        polar   special  non_conservative   
198  seq_2  V       109   P  hydrophobic   special  non_conservative   
104  seq_5  N        59   P        polar   special  non_conservative   
126  seq_5  V       109   P  hydrophobic   special  non_conservative   
11   seq_1  S        24   P        polar   special  non_conservative   
18   seq_1  S        36   G        polar   special  non_conservative   
19   seq_1  N        37   G        polar   special  non_conservative   
20   seq_1  F        38   G     aromatic   special  non_conservative   
32   seq_1  Q        57   G        polar   special  non_conservative   
149  seq_2  S        24   P        polar   special  non_conservative   
156  seq

Mutation Risk Analysis

Mutation-level data from the top redesign candidates were evaluated using simple, biologically interpretable rules.

Mutations were considered more risky when they were:

non-conservative
located in core regions
found in helices or sheets
involved biochemical class switches
affected glycine or proline in structured regions
What was done
Added rule-based mutation risk scoring
Summarized risk across positions
Identified candidate positions for future design constraints

## Mutation Risk Analysis

Mutation-level data from the top redesign candidates were evaluated using **simple, biologically interpretable rules**.

Mutations were considered more risky when they were:

- **Non-conservative**
- Located in **core regions**
- Found in **helices or sheets**
- Involved **biochemical class switches**
- Affected **glycine or proline in structured regions**

---

### What was done

- Added **rule-based mutation risk scoring**
- Summarized **risk across positions**
- Identified **candidate positions for future design constraints**

---

### Why this matters

This step moves the project from **descriptive mutation analysis** toward **iterative protein engineering logic**.

Rather than treating all mutations equally, it identifies positions where future redesign should be **more controlled and biologically informed**.

---

### Important limitation

This is a **heuristic analysis**.

It does **not** replace:

- Energetic modeling  
- Molecular dynamics  
- Functional assays  
- Experimental validation  

***Load constraint positions***

In [80]:
import pandas as pd
import os

# Load constraint positions
possible_paths = [
    "../results/tables/constraint_positions.csv",
    "results/tables/constraint_positions.csv"
]

constraint_file = None
for path in possible_paths:
    if os.path.exists(path):
        constraint_file = path
        break

if constraint_file is None:
    raise FileNotFoundError("constraint_positions.csv not found. Finish Step 22.")

constraint_df = pd.read_csv(constraint_file)

constraint_positions = constraint_df["position"].tolist()

print("Constraint positions:", constraint_positions)
print("Total constraints:", len(constraint_positions))

Constraint positions: [59, 109, 24, 37, 38, 57, 36, 47, 4, 11, 15, 27, 33, 35, 39, 44, 52, 56, 71, 82, 84, 90, 106, 110, 7, 93, 53, 85, 34, 91, 2, 10, 21, 46, 51, 61, 68, 73, 86, 113, 122, 125, 128, 62, 75, 116, 118]
Total constraints: 47


***Load full residue table***

In [81]:
residue_df = pd.read_csv("../data/processed/residue_table.csv")

all_positions = residue_df["residue_number"].tolist()

print("Total residues:", len(all_positions))

Total residues: 129


***Define fixed vs designable positions***

In [82]:
fixed_positions = set(constraint_positions)

designable_positions = [
    pos for pos in all_positions if pos not in fixed_positions
]

print("Fixed positions:", len(fixed_positions))
print("Designable positions:", len(designable_positions))

Fixed positions: 47
Designable positions: 82


***Create ProteinMPNN constraint format-save fixed positions***

In [83]:
import json

output_dir = "../results/design_constraints"
if not os.path.exists(output_dir):
    output_dir = "results/design_constraints"

os.makedirs(output_dir, exist_ok=True)

fixed_file = os.path.join(output_dir, "fixed_positions.json")

with open(fixed_file, "w") as f:
    json.dump({"fixed_positions": list(fixed_positions)}, f, indent=2)

print("Saved:", fixed_file)

Saved: ../results/design_constraints/fixed_positions.json


***Save designable positions***

In [84]:
designable_file = os.path.join(output_dir, "designable_positions.json")

with open(designable_file, "w") as f:
    json.dump({"designable_positions": designable_positions}, f, indent=2)

print("Saved:", designable_file)

Saved: ../results/design_constraints/designable_positions.json


***human-readable table***

In [85]:
constraint_df_sorted = constraint_df.sort_values(
    by=["n_high_risk", "mean_risk_score"],
    ascending=False
)

constraint_df_sorted.to_csv(
    os.path.join(output_dir, "constraint_positions_ranked.csv"),
    index=False
)

print(constraint_df_sorted.head())

   position  n_mutations  n_candidates  max_risk_score  mean_risk_score  \
0        59            3             3               6              6.0   
1       109            3             3               6              6.0   
2        24            3             3               5              5.0   
3        37            3             3               5              5.0   
4        38            3             3               5              5.0   

   n_high_risk environments ss_types  
0            3         core    sheet  
1            3         core    helix  
2            3      surface    sheet  
3            3         core    helix  
4            3         core    helix  


## Constrained Redesign Strategy

Based on mutation risk analysis, selected residue positions were fixed to prevent potentially disruptive substitutions in future design iterations.

### Strategy
- High-risk positions were **fixed**
- Remaining residues were **allowed to mutate**

### Why this matters
This step transforms the workflow from unconstrained sequence generation to **controlled and biologically informed redesign**.

### Outcome
Prepared inputs for a second ProteinMPNN run with structural and biochemical constraints.

***Load fixed positions***

In [86]:
import json
import os

possible_paths = [
    "../results/design_constraints/fixed_positions.json",
    "results/design_constraints/fixed_positions.json"
]

fixed_file = None
for path in possible_paths:
    if os.path.exists(path):
        fixed_file = path
        break

if fixed_file is None:
    raise FileNotFoundError(
        "fixed_positions.json not found. Finish Step 23 first."
    )

with open(fixed_file, "r") as f:
    fixed_data = json.load(f)

fixed_positions = fixed_data["fixed_positions"]

print("Loaded from:", fixed_file)
print("Total fixed positions:", len(fixed_positions))
print("First few:", fixed_positions[:20])

Loaded from: ../results/design_constraints/fixed_positions.json
Total fixed positions: 47
First few: [128, 2, 4, 7, 10, 11, 15, 21, 24, 27, 33, 34, 35, 36, 37, 38, 39, 44, 46, 47]


***Create the ProteinMPNN fixed-position file***

In [87]:
import json
import os

protein_name = "1LYZ_clean"
chain_id = "A"

fixed_position_dict = {
    protein_name: {
        chain_id: fixed_positions
    }
}

output_dir = "../results/design_constraints"
if not os.path.exists(output_dir):
    output_dir = "results/design_constraints"

os.makedirs(output_dir, exist_ok=True)

jsonl_path = os.path.join(output_dir, "proteinmpnn_fixed_positions.jsonl")

with open(jsonl_path, "w") as f:
    f.write(json.dumps(fixed_position_dict) + "\n")

print("Saved:", jsonl_path)
print(json.dumps(fixed_position_dict, indent=2))

Saved: ../results/design_constraints/proteinmpnn_fixed_positions.jsonl
{
  "1LYZ_clean": {
    "A": [
      128,
      2,
      4,
      7,
      10,
      11,
      15,
      21,
      24,
      27,
      33,
      34,
      35,
      36,
      37,
      38,
      39,
      44,
      46,
      47,
      51,
      52,
      53,
      56,
      57,
      59,
      61,
      62,
      68,
      71,
      73,
      75,
      82,
      84,
      85,
      86,
      90,
      91,
      93,
      106,
      109,
      110,
      113,
      116,
      118,
      122,
      125
    ]
  }
}


***on terminal Ran constrained ProteinMPN***

python ProteinMPNN/protein_mpnn_run.py \
  --pdb_path data/design/1LYZ_clean.pdb \
  --out_folder results/design_sequences_round2_constrained \
  --num_seq_per_target 10 \
  --sampling_temp "0.1" \
  --seed 42 \
  --fixed_positions_jsonl results/design_constraints/proteinmpnn_fixed_positions.jsonl


***Parse the constrained round-2 sequences***

In [88]:
from Bio import SeqIO
import pandas as pd
import os

possible_fasta_paths = [
    "../results/design_sequences_round2_constrained/seqs/1LYZ_clean.fa",
    "results/design_sequences_round2_constrained/seqs/1LYZ_clean.fa"
]

round2_fasta = None
for path in possible_fasta_paths:
    if os.path.exists(path):
        round2_fasta = path
        break

if round2_fasta is None:
    raise FileNotFoundError(
        "Could not find round-2 constrained FASTA. Check the output folder."
    )

records = list(SeqIO.parse(round2_fasta, "fasta"))

round2_rows = []
for i, record in enumerate(records):
    round2_rows.append({
        "round": "round2_constrained",
        "name": record.id,
        "sequence": str(record.seq),
        "candidate_rank": i + 1
    })

round2_df = pd.DataFrame(round2_rows)

print("Loaded FASTA:", round2_fasta)
print("Number of sequences:", len(round2_df))
print(round2_df.head())

Loaded FASTA: ../results/design_sequences_round2_constrained/seqs/1LYZ_clean.fa
Number of sequences: 11
                round         name  \
0  round2_constrained  1LYZ_clean,   
1  round2_constrained       T=0.1,   
2  round2_constrained       T=0.1,   
3  round2_constrained       T=0.1,   
4  round2_constrained       T=0.1,   

                                            sequence  candidate_rank  
0  KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRN...               1  
1  MVWGQCELAAALKKHGMDNYRGYSLGNFVCLAKFESNFNNSVTNTN...               2  
2  TVWGQCELAAALKKHGMDNFRGYSLGNFVCLAKFESNFNNSVSNTN...               3  
3  KVFGRCELAAALKKHGMDNFRGYSLGNWVCLAKFESNFNTSVSNTN...               4  
4  KVWGTCELAAALKKHGMDNFRGYSLGNYVCLAKFESNFNNSVSNTN...               5  


***Load the native sequence***

In [91]:
from pathlib import Path

native_seq_path = Path("../data/processed/1LYZ_chain_A_sequence.txt")

if not native_seq_path.exists():
    raise FileNotFoundError(f"Native sequence file not found: {native_seq_path}")

with open(native_seq_path, "r") as f:
    native_sequence = f.read().strip()

print("Loaded native sequence")
print("Length:", len(native_sequence))
print(native_sequence[:50])

Loaded native sequence
Length: 129
KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGS


***Compare constrained round-2 sequences to native***

In [92]:
def compare_to_native(native_seq, designed_seq):
    if len(native_seq) != len(designed_seq):
        raise ValueError(
            f"Length mismatch: native={len(native_seq)}, designed={len(designed_seq)}"
        )

    mutations = []
    for i, (wt, mut) in enumerate(zip(native_seq, designed_seq), start=1):
        if wt != mut:
            mutations.append(f"{wt}{i}{mut}")

    return len(mutations), ";".join(mutations)

comparison_rows = []

for _, row in round2_df.iterrows():
    num_mutations, mutation_string = compare_to_native(native_sequence, row["sequence"])
    
    comparison_rows.append({
        "round": row["round"],
        "name": row["name"],
        "candidate_rank": row["candidate_rank"],
        "sequence": row["sequence"],
        "num_mutations": num_mutations,
        "mutations": mutation_string
    })

round2_compare_df = pd.DataFrame(comparison_rows)

print(round2_compare_df.head())
print(round2_compare_df["num_mutations"].describe())

                round         name  candidate_rank  \
0  round2_constrained  1LYZ_clean,               1   
1  round2_constrained       T=0.1,               2   
2  round2_constrained       T=0.1,               3   
3  round2_constrained       T=0.1,               4   
4  round2_constrained       T=0.1,               5   

                                            sequence  num_mutations  \
0  KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRN...              0   
1  MVWGQCELAAALKKHGMDNYRGYSLGNFVCLAKFESNFNNSVTNTN...             31   
2  TVWGQCELAAALKKHGMDNFRGYSLGNFVCLAKFESNFNNSVSNTN...             32   
3  KVFGRCELAAALKKHGMDNFRGYSLGNWVCLAKFESNFNTSVSNTN...             28   
4  KVWGTCELAAALKKHGMDNFRGYSLGNYVCLAKFESNFNNSVSNTN...             32   

                                           mutations  
0                                                     
1  K1M;F3W;R5Q;M12L;R14K;L17M;W28F;A31L;T40N;Q41S...  
2  K1T;F3W;R5Q;M12L;R14K;L17M;Y20F;W28F;A31L;T40N...  
3  M12L;R14K;L17M;Y20F;A31L;

***Save constrained round-2 comparison table***

In [93]:
import os

output_tables = "../results/tables"
if not os.path.exists(output_tables):
    output_tables = "results/tables"

os.makedirs(output_tables, exist_ok=True)

round2_compare_file = os.path.join(output_tables, "round2_constrained_design_comparison.csv")
round2_compare_df.to_csv(round2_compare_file, index=False)

print("Saved:", round2_compare_file)

Saved: ../results/tables/round2_constrained_design_comparison.csv


***Load round-1 data for comparison***

In [94]:
possible_round1_paths = [
    "../results/tables/selected_candidates.csv",
    "results/tables/selected_candidates.csv"
]

round1_file = None
for path in possible_round1_paths:
    if os.path.exists(path):
        round1_file = path
        break

if round1_file is None:
    raise FileNotFoundError("selected_candidates.csv not found.")

round1_df = pd.read_csv(round1_file)

print("Loaded round 1 file:", round1_file)
print(round1_df.columns.tolist())
print(round1_df.head())

Loaded round 1 file: ../results/tables/selected_candidates.csv
['name', 'sequence', 'num_mutations', 'mutation_positions', 'mutations', 'id', 'file', 'plddt', 'ptm', 'rank_score', 'candidate_rank']
                                                name  \
0  >T=0.1, sample=1, score=0.8716, global_score=0...   
1  >T=0.1, sample=5, score=0.8566, global_score=0...   
2  >T=0.1, sample=2, score=0.8667, global_score=0...   
3  >T=0.1, sample=4, score=0.8506, global_score=0...   
4  >T=0.1, sample=3, score=0.9082, global_score=0...   

                                            sequence  num_mutations  \
0  MKFSRCELARRLKELGMDNLFGYPLGAYVCLAEVTGGGDTSVTRKD...             70   
1  MKFSRCGLAQELKKLGMDNFNGYPLGAWVCLAEVTAGGDTSVSRAY...             67   
2  KKYTRCGLAKELKKLGMDNFNGYPLGYWVCLAWYTAGGDNSKTEKL...             73   
3  KKYSRCELAQTLKKLGMDNFNGYPLGAFVCLAWYTAGGDSSVSRKD...             70   
4  MKYSRCGLAQELKKLGMDNFNGYPLGAYVCLAMVSAGGDNSVSKAD...             76   

                                  muta

***Build round-1 vs round-2 comparison***

In [95]:
round1_summary = round1_df.copy()
round1_summary["round"] = "round1_unconstrained"

round1_keep = [
    col for col in ["round", "name", "candidate_rank", "sequence", "num_mutations"]
    if col in round1_summary.columns
]
round1_summary = round1_summary[round1_keep]

round2_summary = round2_compare_df[[
    "round", "name", "candidate_rank", "sequence", "num_mutations"
]].copy()

combined_df = pd.concat([round1_summary, round2_summary], ignore_index=True)

print(combined_df.head())
print(combined_df["round"].value_counts())

                  round                                               name  \
0  round1_unconstrained  >T=0.1, sample=1, score=0.8716, global_score=0...   
1  round1_unconstrained  >T=0.1, sample=5, score=0.8566, global_score=0...   
2  round1_unconstrained  >T=0.1, sample=2, score=0.8667, global_score=0...   
3  round1_unconstrained  >T=0.1, sample=4, score=0.8506, global_score=0...   
4  round1_unconstrained  >T=0.1, sample=3, score=0.9082, global_score=0...   

   candidate_rank                                           sequence  \
0               1  MKFSRCELARRLKELGMDNLFGYPLGAYVCLAEVTGGGDTSVTRKD...   
1               2  MKFSRCGLAQELKKLGMDNFNGYPLGAWVCLAEVTAGGDTSVSRAY...   
2               3  KKYTRCGLAKELKKLGMDNFNGYPLGYWVCLAWYTAGGDNSKTEKL...   
3               4  KKYSRCELAQTLKKLGMDNFNGYPLGAFVCLAWYTAGGDSSVSRKD...   
4               5  MKYSRCGLAQELKKLGMDNFNGYPLGAYVCLAMVSAGGDNSVSKAD...   

   num_mutations  
0             70  
1             67  
2             73  
3             70  
4  

***Compute summary statistics***

In [96]:
summary_stats = (
    combined_df.groupby("round")
    .agg(
        n_sequences=("sequence", "count"),
        mean_mutations=("num_mutations", "mean"),
        median_mutations=("num_mutations", "median"),
        min_mutations=("num_mutations", "min"),
        max_mutations=("num_mutations", "max")
    )
    .reset_index()
)

print(summary_stats)

                  round  n_sequences  mean_mutations  median_mutations  \
0  round1_unconstrained            5       71.200000              70.0   
1    round2_constrained           11       28.818182              32.0   

   min_mutations  max_mutations  
0             67             76  
1              0             35  


***Save final comparison table***

In [98]:
combined_file = os.path.join(output_tables, "round1_vs_round2_constrained_sequences.csv")
stats_file = os.path.join(output_tables, "round1_vs_round2_constrained_stats.csv")

combined_df.to_csv(combined_file, index=False)
summary_stats.to_csv(stats_file, index=False)

print("Saved:", combined_file)
print("Saved:", stats_file)

Saved: ../results/tables/round1_vs_round2_constrained_sequences.csv
Saved: ../results/tables/round1_vs_round2_constrained_stats.csv


***plot***